In [1]:
# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
ss = hf.settings_dict()

In [2]:
for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:
        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-vol.stc"
        stc = mne.read_source_estimate(hilbert_stc_file)

        # load hilbert references
        retina_file = Path(ss['hilbert_dir']) / subject / event_name / f"{subject}-event-{event_name}-reference.csv"

        # Read the CSV file
        df = pd.read_csv(retina_file)

        reference_voxel = df[df.axes[1][2]]
        reference_voxel_squeezed = reference_voxel.values.squeeze()
        reference_retina = df["retina_phase"]
        reference_retina_squeezed = reference_retina.values.squeeze()

        # Sanity check: should match STC timepoints
        assert len(reference_voxel_squeezed) == stc.data.shape[1], (
            f"Length mismatch: CSV has {len(reference_voxel_squeezed)} timepoints, "
            f"STC has {stc.data.shape[1]}"
        )
        assert len(reference_retina_squeezed) == stc.data.shape[1], (
            f"Length mismatch: CSV has {len(reference_retina_squeezed)} timepoints, "
            f"STC has {stc.data.shape[1]}"
        )

        # subtract phases from stc
        phase_diff_voxel = stc.data - reference_voxel_squeezed[np.newaxis, :]
        phase_diff_voxel_wrapped = np.angle(np.exp(1j * phase_diff_voxel))
        phase_diff_wrapped_voxel_stc = stc.copy()
        phase_diff_wrapped_voxel_stc.data = phase_diff_voxel_wrapped
        phase_diff_voxel_stc = stc.copy()
        phase_diff_voxel_stc.data = phase_diff_voxel

        phase_diff_retina = stc.data - reference_retina_squeezed[np.newaxis, :]
        phase_diff_retina_wrapped = np.angle(np.exp(1j * phase_diff_retina))
        phase_diff_wrapped_retina_stc = stc.copy()
        phase_diff_wrapped_retina_stc.data = phase_diff_retina_wrapped
        phase_diff_retina_stc = stc.copy()
        phase_diff_retina_stc.data = phase_diff_retina

        phase_diff_wrapped_voxel_stc.save(save_dir / f"{subject}-event-{event_name}-hilbert-vox-ref-wrp-vol.stc", overwrite=True)
        phase_diff_voxel_stc.save(save_dir / f"{subject}-event-{event_name}-hilbert-ret-ref-vol.stc" , overwrite=True)
        phase_diff_wrapped_retina_stc.save(save_dir / f"{subject}-event-{event_name}-hilbert-ret-ref-wrp-vol.stc" , overwrite=True)
        phase_diff_retina_stc.save(save_dir / f"{subject}-event-{event_name}-hilbert-vox-ref-vol.stc" , overwrite=True)


        del phase_diff_wrapped_voxel_stc
        del phase_diff_wrapped_retina_stc
        del phase_diff_voxel_stc
        del phase_diff_retina_stc
        del stc
        gc.collect()


loading dataset for subject:  0005_3SJ
Writing STC to disk...
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
Writing STC to disk...
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
Writing STC to disk...
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
Writing STC to disk...
[done]
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
Writing STC to disk...
[done]
Overwriting exis